# 09 · Аудит данных XLS-R
Без обучения и автоматической загрузки ASVspoof 5.

In [ ]:
import subprocess
import sys
from pathlib import Path

repo=Path('/content/audio-deepfake-detector')
if not (repo/'.git').exists(): subprocess.run(['git','clone','https://github.com/Unvand/audio-deepfake-detector.git',str(repo)],check=True)
else: subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(repo/'requirements-colab-train.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<5'],check=True)
sys.path.insert(0,str(repo))
commit=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
from google.colab import drive

drive.mount('/content/drive')
base=Path('/content/drive/MyDrive/VKR_RuASD')
prepared=base/'prepared_v1'
print('Git commit:',commit)

In [ ]:
from audio_deepfake_detector.src.xlsr_training import XLSRConfig, audit_manifest

frame,summary=audit_manifest(prepared/'ruasd_segments_manifest.csv')
display(summary)
print('Пересечения sample_id:',frame.groupby('sample_id').split.nunique().gt(1).sum())
print('Идентификаторы дикторов:',[c for c in frame if 'speaker' in c.lower()])
print('Конфигурация:',XLSRConfig())
asv=base/'ASVspoof5'
print('ASVspoof 5 на Drive:',asv.exists())
if asv.exists():
 print('Имена файлов верхнего уровня:',[p.name for p in list(asv.iterdir())[:30]])
 print('Перед объединением: проверьте официальный протокол, лицензию, метки, дикторов и split.')

In [ ]:
RUN_SAVE_AUDIT=False
if RUN_SAVE_AUDIT:
 out=base/'training_xlsr_v1/audit';out.mkdir(parents=True,exist_ok=True)
 summary.to_csv(out/'ruasd_split_summary.csv',index=False)
 print('Аудит сохранён:',out)